# Supplementary Fig. 1: behavior grid (panels a and c, example columns)

Scores for panel c come from `sf1c_scores.py` (this notebook's own score column for that row was replaced in the figure).

# Supplementary Figure S1 — Behavior grid

Five tasks (rows) $\times$ four views (columns):

| column | content |
|---|---|
| **Task schematic** | cartoon of the task |
| **Target behavior** | the ideal / target behavior |
| **Example model behavior** | the actual behavior of a representative best-performing rule (real simulation) |
| **Scores across rules** | best performance of all seven rules, normalized within the task |

**Rows and score metric** (all read from the aggregated sweep tensors in `data/sweeps/`):

1. **Feature selectivity** — selectivity $\chi$ (higher better)
2. **Slow feature selectivity** — maximin $\chi$ over the two feature-timescale conditions
3. **Costly stochasticity** (matching pennies) — 3-bit action entropy (bits, max 3)
4. **Baited bandit** — probability matching on the *baited* bandit: $-|\text{ou index}|$ (0 = perfect matching)
5. **Unbaited bandit** — probability matching on the *unbaited* bandit: $-|\text{pm index}|$

**Example-rule note.** The *scores* column ranks the seven rules by the metric above. For the two
matching rows the example-behavior panel instead shows **PC**, the representative matcher used in the
main text: the *signed* matching index can be driven to $\approx 0$ by a degenerate near-flat policy
(see the docstring of `scorers.score_probmatching`), so the numeric best-by-$|\text{index}|$ rule is not
always the clearest illustration of matching. The example rule for every panel is printed and labelled.

Run top-to-bottom. Regenerating the SVG/PNG takes ~1 min on CPU (the two 50k-step matching sweeps dominate).


In [ ]:
import os, sys, numpy as np
import matplotlib as mpl, matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import FancyArrowPatch, Circle, FancyBboxPatch, Rectangle
import jax; jax.config.update('jax_platform_name', 'cpu')
import jax.numpy as jnp

AGG = '../../../data/sweeps'   # aggregated sweep tensors
print('aggregated:', AGG, '->', os.path.isdir(AGG))

try: mpl.rcParams['font.family'] = 'Arial'
except Exception: pass
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['xtick.direction'] = 'in'    # ticks point into the axes
mpl.rcParams['ytick.direction'] = 'in'

BLUE, GREY, LGREY, ACC = '#4A90D9', '#6d6e71', '#d1d3d4', '#E4572E'
SEED_COLORS = ['#3B6FB6', '#8E44AD', '#17A589']   # 3 seeds, distinct hues
MATCH_RULES  = ['COV', 'BCM', 'PC', 'OJA']        # rules shown in both matching plots
MATCH_COLORS = ['#3B6FB6', '#8E44AD', '#17A589', '#C0392B']

def nature_ax(ax, xlabel='', ylabel='', nx=4, ny=4, title=''):
    for sp in ('top', 'right'): ax.spines[sp].set_visible(False)
    for sp in ('left', 'bottom'): ax.spines[sp].set_linewidth(0.4)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(nx))
    ax.yaxis.set_major_locator(mticker.MaxNLocator(ny))
    ax.tick_params(labelsize=6, width=0.4, length=2)
    if xlabel: ax.set_xlabel(xlabel, fontsize=8)
    if ylabel: ax.set_ylabel(ylabel, fontsize=8)
    if title: ax.set_title(title, fontsize=8)


In [ ]:
# ---------- scores engine: best rule + best hyperparameters per task ----------
# Sweep grids (must match ssb/scripts/run_sweep.py).
ETAS   = np.logspace(-6, -1, 6)
TAUS   = np.logspace(-3, -1, 4)
ALPHAS = np.logspace(-5, 0, 15)
BETAS  = np.logspace(-5, 0, 15)
RULES     = ['PC', 'HR', 'OJA', 'COV', 'BCM', 'LC', 'HZ']
TAU_RULES = {'COV', 'BCM', 'LC', 'HZ'}

def _to_etau(a):
    return a[:, None] if a.ndim == 3 else a

def perf_field(task, rule, size):
    """Higher-is-better performance with axes (eta, tau, alpha, beta)."""
    if task == 'selectivity':
        return _to_etau(np.load(f'{AGG}/selectivity_{rule}_{size}.npy'))
    if task == 'slowfeature':
        return _to_etau(np.min(np.load(f'{AGG}/slowfeature_{rule}_{size}.npy'), axis=-1))
    if task == 'matchingpennies':
        return _to_etau(np.load(f'{AGG}/matchingpennies_{rule}_{size}.npy')[..., 2])
    if task == 'baited':
        return -np.abs(_to_etau(np.load(f'{AGG}/bandit_bait_{rule}_{size}.npy')))
    if task == 'unbaited':
        return -np.abs(_to_etau(np.load(f'{AGG}/bandit_unbait_{rule}_{size}.npy')[..., 0]))
    raise ValueError(task)

def best_hparams(task, rule, size):
    P = perf_field(task, rule, size); flat = P.reshape(-1)
    if np.all(np.isnan(flat)): return None
    k = int(np.nanargmax(flat)); e, t, a, b = np.unravel_index(k, P.shape)
    tau = float(TAUS[t]) if rule in TAU_RULES else 0.01
    return dict(rule=rule, eta=float(ETAS[e]), tau=tau,
                alpha=float(ALPHAS[a]), beta=float(BETAS[b]), perf=float(flat[k]))

def rule_scores(task, size):
    return {r: (best_hparams(task, r, size) or {'perf': np.nan})['perf'] for r in RULES}

# ---- probability matching: score = MAX PM index (strongest exploitation regime) ----
def mm_field(task, rule, size):
    if task == 'baited':
        a = np.load(f'{AGG}/bandit_bait_{rule}_{size}.npy')
    else:
        a = np.load(f'{AGG}/bandit_unbait_{rule}_{size}.npy')[..., 0]
    return a[:, None] if a.ndim == 3 else a

def matching_max_score(task, rule, size='small'):
    return float(np.nanmax(mm_field(task, rule, size)))

def matching_max_cell(task, rule, size='small'):
    a = mm_field(task, rule, size); flat = a.reshape(-1)
    k = int(np.nanargmax(flat)); e, t, al, b = np.unravel_index(k, a.shape)
    tau = float(TAUS[t]) if rule in TAU_RULES else 0.01
    return dict(rule=rule, eta=float(ETAS[e]), tau=tau, alpha=float(ALPHAS[al]),
                beta=float(BETAS[b]), index=float(flat[k]))

def matching_shared_cell(rule, size='small'):
    """One cell per rule for BOTH bandits (joint-best of baited+unbaited index)."""
    b = mm_field('baited', rule, size); u = mm_field('unbaited', rule, size)
    joint = b + u; k = int(np.nanargmax(joint.reshape(-1)))
    e, t, al, be = np.unravel_index(k, joint.shape)
    tau = float(TAUS[t]) if rule in TAU_RULES else 0.01
    return dict(rule=rule, eta=float(ETAS[e]), tau=tau, alpha=float(ALPHAS[al]),
                beta=float(BETAS[be]), bait_idx=float(b[e, t, al, be]),
                unbait_idx=float(u[e, t, al, be]))

def best_rule(task, size):
    s = rule_scores(task, size)
    return max(s, key=lambda r: (-np.inf if np.isnan(s[r]) else s[r]))

def worst_rule(task, size):
    s = rule_scores(task, size)
    return min(s, key=lambda r: (np.inf if np.isnan(s[r]) else s[r]))

# --- matching pennies: two entropies (vs computer opponent / vs random reward) ---
# Sweep conditions were [play_mp, bandit_unbait(0.5), play_mp] -> entropy indices
# [0, 1, 2]. Computer-opponent entropy = index 2 (last play_mp); random-reward
# entropy = index 1. "Separation" = computer - random (big = stochastic vs opponent
# but biased under random reward, like the animal data).
def mp_fields(rule, size):
    a = np.load(f'{AGG}/matchingpennies_{rule}_{size}.npy')   # (...,3)
    comp, rand = a[..., 2], a[..., 1]
    if comp.ndim == 3:                                        # insert tau axis
        comp, rand = comp[:, None], rand[:, None]
    return comp, rand                                        # (eta,tau,alpha,beta)

def mp_best_cell(rule, size):
    comp, rand = mp_fields(rule, size); gap = comp - rand
    k = int(np.nanargmax(gap.reshape(-1)))
    e, t, a, b = np.unravel_index(k, gap.shape)
    tau = float(TAUS[t]) if rule in TAU_RULES else 0.01
    return dict(rule=rule, eta=float(ETAS[e]), tau=tau, alpha=float(ALPHAS[a]),
                beta=float(BETAS[b]), comp=float(comp[e, t, a, b]),
                rand=float(rand[e, t, a, b]), gap=float(gap[e, t, a, b]))

def mp_scores(size):
    return {r: mp_best_cell(r, size) for r in RULES}

def mp_best_rule(size):
    s = mp_scores(size)
    return max(s, key=lambda r: s[r]['gap'])

def goodness_within_task(scores):
    """Min-max normalize the per-rule (higher-is-better) scores to [0,1]."""
    v = np.array([scores[r] for r in RULES], float)
    lo, hi = np.nanmin(v), np.nanmax(v)
    g = np.where(np.isfinite(v), (v - lo) / (hi - lo + 1e-12), np.nan)
    return g


In [ ]:
# ---------- real model runs (mirror ssb/scripts/run_sweep.py env setup) ----------
from ssb.models import ssb
from ssb.envs import streams, bandits
from ssb.envs import matchingpennies as mp
from ssb.scripts.run import run_env
from ssb.utils.batch_inits import make_default_hps, batchify_hps, expand_hps_batch
from ssb.utils.architectures import get_sizes, get_n_motor

def _hps_for(bh, env, arch='small', seeds=12, seed=17):
    hps = make_default_hps(seed)
    hps['eta_a'] = jnp.array([bh['eta']]); hps['tau'] = jnp.array([bh['tau']])
    nm = get_n_motor(env)
    idim = {'selectivity': 37, 'slowfeature': 2, 'matchingpennies': 1,
            'bandit_bait': 1, 'bandit_unbait': 1}[env]
    hps['sizes'] = jnp.array(get_sizes(arch, idim, nm))
    hb = batchify_hps(hps, seeds)
    hb = expand_hps_batch(hb, jnp.array([bh['alpha']]), 'alpha')
    hb = expand_hps_batch(hb, jnp.array([bh['beta']]),  'beta')
    return hb

_FILTERS = streams.make_filters(37, 15, sharpness=20)   # 15 oriented filters, dim 37

def _tuning_from_ckpt(ckpt):
    """Per-seed tuning curve (response to each of the 15 filters)."""
    acts_f, w_f, t_f = ckpt[1], ckpt[3], ckpt[4]
    pad = acts_f.shape[-1] - _FILTERS.shape[-1]
    stim_set = jnp.pad(jnp.asarray(_FILTERS), ((0, 0), (0, pad)))
    def net_tuning(a_s, w_s, tr_s):
        def single(s):
            a = jnp.zeros_like(a_s)
            for _ in range(3): a = s + jnp.matmul(a, w_s * tr_s)
            return jnp.maximum(0, a)[-1]
        return jax.vmap(single)(stim_set)
    return np.asarray(jax.vmap(net_tuning)(acts_f, w_f, t_f))   # (seeds, 15)

def _selectivity(tuning):
    rel = tuning[:, [4, 11]].mean(1); irr = np.delete(tuning, [4, 11], axis=1).mean(1)
    return (rel - irr) / (rel + irr + 1e-6)

def _run_sel(bh, T, seeds):
    hb = _hps_for(bh, 'selectivity', seeds=seeds)
    _, ckpt = run_env(hb, bh['rule'], streams.sample_inputs, jnp.asarray(_FILTERS),
                      T, settle_time=1, env_state=jnp.array([0., 0., 1.]), n_motor=1)
    return _tuning_from_ckpt(ckpt)

def run_selectivity(bh, T=4000, seeds=24):
    """Best-seed tuning curve + its selectivity (for the example panel)."""
    tuning = _run_sel(bh, T, seeds)
    sel = _selectivity(tuning); best = int(np.nanargmax(sel))
    return dict(kind='tuning', tuning=tuning[best], rel_idx=[4, 11], sel=float(sel[best]))

def _sel_general(tun):
    """General single-unit selectivity (à la BCM-1982): 1 - mean/max over the
    tuning curve, in [0,1]. 1 = responds to one orientation; 0 = flat. Works on
    any trailing-15 array (broadcasts over leading axes)."""
    return 1.0 - tun.mean(-1) / (tun.max(-1) + 1e-9)

def _tuning_timecourse(bh, Ts, seeds):
    """Per-seed tuning curves at each t in Ts -> (nT, seeds, 15)."""
    return np.array([_run_sel(bh, T, seeds) for T in Ts])

def _pick_diverse_peaks(tuning, n, min_sep=2):
    """Pick n seeds with clear tuning peaks located at DIFFERENT orientations.
    Rank by peakedness (max/mean); greedily keep seeds whose peak location is at
    least `min_sep` bins from those already chosen; relax if too few qualify."""
    peak = tuning.argmax(1)
    peakedness = tuning.max(1) / (tuning.mean(1) + 1e-9)
    order = list(np.argsort(peakedness)[::-1])
    chosen, locs = [], []
    for s in order:
        if all(abs(int(peak[s]) - l) >= min_sep for l in locs):
            chosen.append(int(s)); locs.append(int(peak[s]))
        if len(chosen) == n:
            break
    for s in order:                       # fallback fill (keep most-peaked)
        if len(chosen) == n:
            break
        if int(s) not in chosen:
            chosen.append(int(s))
    return chosen

def run_sel_seeds(bh, n_show=3, Ts=(10, 25, 50, 100, 150, 250, 400, 600, 800, 1000),
                  seeds=48, min_sep=2):
    """Selectivity trajectories (top) + final tuning curves (bottom) for n_show
    seeds chosen to have obvious peaks at different orientations. The trajectory
    is the general (location-free) selectivity so a seed peaking anywhere rises.
    Used for both the LC target and the PC example."""
    TUN = _tuning_timecourse(bh, Ts, seeds)          # (nT, seeds, 15)
    tun_final = TUN[-1]
    idx = _pick_diverse_peaks(tun_final, n_show, min_sep=min_sep)
    sel_traj = _sel_general(TUN)                     # (nT, seeds)
    return dict(ts=np.array(Ts), sel=np.clip(sel_traj[:, idx], 0, 1),
                tuning=tun_final[idx], chis=_sel_general(tun_final)[idx],
                peaks=[int(tun_final[i].argmax()) for i in idx], rel_idx=[4, 11])

def _smooth(x, w=21):
    k = np.ones(w) / w
    return np.convolve(x, k, mode='same')

def run_slowfeature(bh, T=1500, seeds=12):
    hb = _hps_for(bh, 'slowfeature', seeds=seeds)
    log, _ = run_env(hb, bh['rule'], streams.stream_slow_feature, jnp.array([0.05, 10.]),
                     T, settle_time=1, env_state=jnp.array([0.]), n_motor=1)
    stim = np.asarray(log[1]); out = np.asarray(log[2])           # (T, seeds)
    sf = stim[:, 0]                                               # slow feature (shared)
    sf_c = sf - sf.mean()
    # pick the seed whose (smoothed) output best tracks the slow feature
    corrs = [abs(np.corrcoef(_smooth(out[:, s]) - _smooth(out[:, s]).mean(), sf_c)[0, 1])
             for s in range(out.shape[1])]
    b = int(np.nanargmax(corrs))
    return dict(kind='slow', stim=sf, out=out[:, b], corr=float(corrs[b]))

def run_matchingpennies(bh, T=1500, seeds=12):
    hb = _hps_for(bh, 'matchingpennies', seeds=seeds)
    log, _ = run_env(hb, bh['rule'], mp.play_mp, jnp.array([0.05]), T,
                     settle_time=1, env_state=mp.init_mp_state(100), n_motor=2)
    acts = np.asarray(log[0]).astype(int)
    ent = float(np.asarray(mp.get_3bit_entropy(jnp.asarray(acts[:, 0]))))
    return dict(kind='mp', actions=acts[:, 0], entropy=ent)

def _matching_curve(bh, envfn, envname, T=50000, window=(25000, 50000), seeds=12):
    prob_num = 7
    probs = np.vstack([np.linspace(0., .3, prob_num), np.linspace(.3, 0., prob_num)]).T
    target = np.linspace(1., 0., prob_num)
    hb = _hps_for(bh, envname, seeds=seeds); per = []
    for p0, p1 in probs:
        log, _ = run_env(hb, bh['rule'], envfn, jnp.array([float(p0), float(p1)]),
                         T, settle_time=1, env_state=None, n_motor=2)
        per.append(np.asarray(log[0])[window[0]:window[1]].mean(0))   # per-seed choice ratio
    per = np.array(per)                                               # (7, seeds)
    # Seed-average washes out matching when seeds settle on different arms
    # (the signed index cancels). Show a REPRESENTATIVE session: the seed whose
    # curve has the median correlation with the reward ratio.
    corr = np.array([np.corrcoef(per[:, s], target)[0, 1] for s in range(per.shape[1])])
    rep = int(np.argsort(np.nan_to_num(corr, nan=-2.0))[len(corr) // 2])
    return dict(kind='match', reward_ratio=target, choice_ratio=per.mean(1),
                choice_rep=per[:, rep], choice_sd=per.std(1))

def run_baited(bh, **kw):   return _matching_curve(bh, bandits.bandit_bait,   'bandit_bait', **kw)
def run_unbaited(bh, **kw): return _matching_curve(bh, bandits.bandit_unbait, 'bandit_unbait', **kw)

def _entropy_3bit_np(a):
    a = np.asarray(a).astype(int)
    if len(a) < 3:
        return 0.0
    s = a[:-2] * 4 + a[1:-1] * 2 + a[2:]
    c = np.bincount(s, minlength=8); p = c / c.sum(); p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

def _rolling_3bit(a, W=100):
    T = len(a); out = np.full(T, np.nan)
    for t in range(W, T + 1):
        out[t - 1] = _entropy_3bit_np(a[t - W:t])
    return out

MATCH_CACHE = '_matching_cache.npz'   # precomputed seed-mean curves + slopes

def _match_curve(task, rule, T=50000, win=(25000, 50000), seeds=32, seed=825):
    """Seed-mean choice-vs-reward curve at the rule's MAX-index (exploitation)
    cell. Uses the sweep seed (825) so seeds are not a fixating subsample."""
    bh = matching_shared_cell(rule, 'small')
    envfn = bandits.bandit_bait if task == 'baited' else bandits.bandit_unbait
    probs = np.vstack([np.linspace(0, .3, 7), np.linspace(.3, 0, 7)]).T
    hps = make_default_hps(seed)
    hps['eta_a'] = jnp.array([bh['eta']]); hps['tau'] = jnp.array([bh['tau']])
    hps['sizes'] = jnp.array(get_sizes('small', 1, 2))
    hb = batchify_hps(hps, seeds)
    hb = expand_hps_batch(hb, jnp.array([bh['alpha']]), 'alpha')
    hb = expand_hps_batch(hb, jnp.array([bh['beta']]), 'beta')
    per = []
    for p0, p1 in probs:
        log, _ = run_env(hb, rule, envfn, jnp.array([float(p0), float(p1)]),
                         T, settle_time=1, env_state=None, n_motor=2)
        per.append(np.asarray(log[0])[win[0]:win[1]].mean(0))
    return np.array(per).mean(1)

def get_matching(task):
    """Load cached 7-rule matching curves/slopes; compute+cache if missing."""
    rr = np.linspace(1, 0, 7)
    if os.path.exists(MATCH_CACHE):
        d = dict(np.load(MATCH_CACHE))
        if f'{task}_curves' in d:
            return dict(rr=d['reward_ratio'], curves=d[f'{task}_curves'],
                        slopes=d[f'{task}_slopes'], rules=[str(r) for r in d['rules']])
    curves = np.array([_match_curve(task, r) for r in RULES])
    slopes = np.array([np.polyfit(rr, c, 1)[0] for c in curves])
    d = dict(np.load(MATCH_CACHE)) if os.path.exists(MATCH_CACHE) else {}
    d[f'{task}_curves'] = curves; d[f'{task}_slopes'] = slopes
    d['reward_ratio'] = rr; d['rules'] = np.array(RULES)
    np.savez(MATCH_CACHE, **d)
    return dict(rr=rr, curves=curves, slopes=slopes, rules=list(RULES))

def run_mp_behavior(bh, T=400, W=100, seeds=20):
    """Two conditions at the best-gap cell (like the animal figure):
    vs computer opponent (play_mp) -> stays stochastic; vs random reward
    p=0.5 (bandit_unbait) -> becomes biased. Returns action rasters + rolling
    3-bit entropy traces for a representative seed of each."""
    hb = _hps_for(bh, 'matchingpennies', seeds=seeds)
    lc, _ = run_env(hb, bh['rule'], mp.play_mp, jnp.array([0.05]), T, settle_time=1,
                    env_state=mp.init_mp_state(100), n_motor=2)
    lr, _ = run_env(hb, bh['rule'], bandits.bandit_unbait, jnp.array([0.5, 0.5]), T,
                    settle_time=1, env_state=None, n_motor=2)
    ca = np.asarray(lc[0]).astype(int); ra = np.asarray(lr[0]).astype(int)  # (T, seeds)
    ce = np.array([_rolling_3bit(ca[:, s], W) for s in range(seeds)])
    re = np.array([_rolling_3bit(ra[:, s], W) for s in range(seeds)])
    cs = int(np.nanargmax(np.nanmean(ce, 1)))    # most stochastic vs opponent
    rs = int(np.nanargmin(re[:, -1]))            # most biased under random reward
    return dict(comp_act=ca[:, cs], comp_ent=ce[cs],
                rand_act=ra[:, rs], rand_ent=re[rs], W=W)

print('run helpers ready')


In [ ]:
# ---------- column 1: task schematics (left empty on purpose) ----------
def draw_schematic(ax, task):
    # Schematics are drawn by hand later; keep these panels blank.
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')


In [ ]:
# ---------- column 2: target (ideal) behavior ----------
def draw_sel_stack(ax, o, colors, xlab='stimulus', tag=''):
    """Stacked selectivity view (lines only): (a) selectivity(t) trajectories
    over (b) final tuning curves, one line per seed. Shared by the LC target
    and the PC example so they read the same way."""
    ax.axis('off')
    top = ax.inset_axes([0.30, 0.60, 0.66, 0.38])
    bot = ax.inset_axes([0.30, 0.10, 0.66, 0.38])
    for j in range(o['sel'].shape[1]):
        top.plot(o['ts'], o['sel'][:, j], color=colors[j], lw=1.1)
    top.set_ylim(0, 1.0)
    nature_ax(top, '', '', nx=3, ny=2); top.set_xlabel('t', fontsize=7, labelpad=1)
    top.set_ylabel('selectivity', fontsize=6, labelpad=1)
    if tag:
        top.text(0.99, 0.06, tag, transform=top.transAxes, fontsize=6, ha='right',
                 va='bottom', color=GREY)
    x = np.arange(o['tuning'].shape[1])
    for j in range(o['tuning'].shape[0]):
        yj = o['tuning'][j] / (o['tuning'][j].max() + 1e-9)
        bot.plot(x, yj, color=colors[j], lw=1.1, zorder=3)
    bot.set_ylim(0, 1.0)
    nature_ax(bot, '', '', nx=3, ny=2); bot.set_xticks([])
    bot.set_xlabel(xlab, fontsize=7, labelpad=1); bot.set_ylabel('response', fontsize=6, labelpad=1)

def draw_target(ax, task, d=None):
    if task == 'selectivity':
        # BCM-1982 Fig.4 style, from LC directly (3 seeds): selectivity buildup
        # over t (a) above the final tuning curves (b).
        draw_sel_stack(ax, d['lc_target'], SEED_COLORS, xlab='orientation', tag='LC')
        return
    if task == 'slowfeature':
        t = np.linspace(0, 1, 200); slow = np.sin(2 * np.pi * t)
        ax.plot(t, slow, color=ACC, lw=1.1)
        nature_ax(ax, 'time', 'output', nx=3, ny=3); ax.set_xticks([]); ax.set_yticks([])
    elif task == 'matchingpennies':
        ax.bar([0, 1], [0.5, 0.5], color=ACC, width=0.6)
        nature_ax(ax, 'action', 'P', nx=2, ny=3)
        ax.set_xticks([0, 1]); ax.set_xticklabels(['A', 'B'], fontsize=6); ax.set_ylim(0, 1)
        ax.axhline(0.5, color=GREY, lw=0.4, ls=':')
        ax.text(0.5, 0.9, 'max entropy', ha='center', transform=ax.transAxes, fontsize=6, color=GREY)
    elif task in ('baited', 'unbaited'):
        ax.plot([0, 1], [0, 1], color=ACC, lw=1.2)
        ax.plot([0, 1], [0, 1], color='k', lw=0.5, ls='--', alpha=0.4)
        nature_ax(ax, 'reward ratio', 'choice ratio', nx=3, ny=3)
        ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect('equal')
        ax.set_xticks([0, .5, 1]); ax.set_yticks([0, .5, 1])
        ax.text(0.5, 0.06, 'matching', ha='center', transform=ax.transAxes, fontsize=6, color=GREY)


In [ ]:
# ---------- column 3: example model behavior (best + worst rule, real runs) ----------
def _mp_raster(ax, a, pos):
    r = ax.inset_axes(pos); t = np.arange(len(a))
    r.vlines(t[a == 1], 0.55, 1.0, color='k', lw=0.3)   # H (action 1)
    r.vlines(t[a == 0], 0.0, 0.45, color='k', lw=0.3)   # T (action 0)
    r.set_xlim(0, len(a)); r.set_ylim(-0.1, 1.1)
    r.set_xticks([]); r.set_yticks([0.22, 0.78]); r.set_yticklabels(['T', 'H'], fontsize=4)
    for sp in r.spines.values(): sp.set_visible(False)
    r.tick_params(length=0)

def _mp_entrace(ax, e, pos, label, xlabel=False):
    g = ax.inset_axes(pos); g.plot(np.arange(len(e)), e, color=ACC, lw=0.8)
    g.set_ylim(0, 3); g.set_xlim(0, len(e))
    for sp in ('top', 'right'): g.spines[sp].set_visible(False)
    for sp in ('left', 'bottom'): g.spines[sp].set_linewidth(0.4)
    g.set_yticks([0, 3]); g.tick_params(labelsize=5, width=0.4, length=2)
    if xlabel:
        g.set_xlabel('Trials', fontsize=7, labelpad=1)
        g.xaxis.set_major_locator(mticker.MaxNLocator(3))
    else:
        g.set_xticks([])
    g.text(1.02, 0.5, label, transform=g.transAxes, fontsize=5, color=ACC,
           rotation=270, va='center', ha='left')

def draw_mp_example(ax, beh):
    ax.axis('off')
    _mp_raster(ax, beh['comp_act'], [0.30, 0.82, 0.60, 0.08])
    _mp_entrace(ax, beh['comp_ent'], [0.30, 0.60, 0.60, 0.20], 'Stochastic')
    _mp_raster(ax, beh['rand_act'], [0.30, 0.34, 0.60, 0.08])
    _mp_entrace(ax, beh['rand_ent'], [0.30, 0.12, 0.60, 0.20], 'Biased', xlabel=True)
    ax.text(0.30, 1.02, 'Computer opponent', transform=ax.transAxes, fontsize=5.5, va='bottom')
    ax.text(0.30, 0.54, 'p(reward) = 0.5', transform=ax.transAxes, fontsize=5.5, va='bottom')
    ax.text(0.12, 0.5, '3-bit entropy', transform=ax.transAxes, fontsize=6.5,
            rotation=90, va='center', ha='center')

def draw_matching_examples(ax, exs):
    """Fixed set of rules (COV, BCM, PC, OJA) as choice-vs-reward curves."""
    ax.plot([0, 1], [0, 1], color='k', lw=0.5, ls='--', alpha=0.4, zorder=1)
    for j, e in enumerate(exs):
        ax.plot(e['rr'], e['curve'], color=MATCH_COLORS[j], lw=1.2, zorder=3)
        ax.text(0.03, 0.99 - 0.11 * j, f"{SCORE_LABELS[e['rule']]} ({e['score']:+.2f})",
                transform=ax.transAxes, ha='left', va='top', fontsize=4.5, color=MATCH_COLORS[j])
    nature_ax(ax, 'reward ratio', 'choice ratio', nx=3, ny=3)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect('equal')
    ax.set_xticks([0, .5, 1]); ax.set_yticks([0, .5, 1])

def draw_example(ax, task, d):
    if task == 'selectivity':
        # PC across 3 seeds (lines only), laid out like the LC target.
        draw_sel_stack(ax, d['pc_ex'], SEED_COLORS, xlab='stimulus', tag='PC')
        return
    if task == 'matchingpennies':
        # Animal-data style: vs computer opponent (stays stochastic) and under
        # random reward p=0.5 (becomes biased), for the best-gap rule.
        draw_mp_example(ax, d['mp_beh'])
        return
    if task in ('baited', 'unbaited'):
        draw_matching_examples(ax, d['examples'])
        return
    ob, ow = d['out_best'], d['out_worst']
    kind = ob['kind']
    if kind != 'mp':      # mp labels rows + names in its entropy line
        ax.text(0.03, 0.99, f"best: {d['best']}", transform=ax.transAxes,
                ha='left', va='top', fontsize=5, color=BLUE, zorder=6)
        ax.text(0.03, 0.86, f"worst: {d['worst']}", transform=ax.transAxes,
                ha='left', va='top', fontsize=5, color=GREY, zorder=6)
    if kind == 'tuning':
        x = np.arange(15)                                        # each curve to own max
        for idx in ob['rel_idx']:
            ax.axvline(idx, color=ACC, lw=0.4, ls=':', zorder=1)
        ax.plot(x, ow['tuning'] / (ow['tuning'].max() + 1e-9), color=GREY, lw=1.0,
                marker='o', ms=2.5, zorder=2)
        ax.plot(x, ob['tuning'] / (ob['tuning'].max() + 1e-9), color=BLUE, lw=1.0,
                marker='o', ms=2.5, zorder=3)
        nature_ax(ax, 'stimulus', 'response', nx=3, ny=3); ax.set_xticks([]); ax.set_ylim(0, 1.1)
    elif kind == 'slow':
        t = np.arange(len(ob['stim']))
        s = ob['stim'] - ob['stim'].mean(); s = s / (np.abs(s).max() + 1e-9)
        def norm_sm(o):
            u = _smooth(o['out']); u = u - u.mean(); return u / (np.abs(u).max() + 1e-9)
        ax.plot(t, s, color='k', lw=1.2, zorder=1)                 # slow feature (target)
        ax.plot(t, norm_sm(ow), color=GREY, lw=0.9, zorder=2)      # worst output
        ax.plot(t, norm_sm(ob), color=BLUE, lw=1.1, zorder=3)      # best output
        nature_ax(ax, 'time', 'norm.', nx=3, ny=3); ax.set_xticks([]); ax.set_yticks([])
    elif kind == 'mp':
        img = np.vstack([ow['actions'][:120], ob['actions'][:120]])  # row0 worst, row1 best
        ax.imshow(img, aspect='auto', cmap='Greys', vmin=0, vmax=1,
                  interpolation='nearest', extent=[0, 120, 0, 2])
        ax.set_yticks([0.5, 1.5]); ax.set_yticklabels(['worst', 'best'], fontsize=5.5)
        ax.set_xlabel('trial', fontsize=8)
        ax.tick_params(labelsize=6, width=0.4, length=2)
        for sp in ('top', 'right', 'left'): ax.spines[sp].set_visible(False)
        ax.spines['bottom'].set_linewidth(0.4)
        ax.text(0.5, 1.03,
                f"3-bit entropy  {d['best']} {ob['entropy']:.2f} / {d['worst']} {ow['entropy']:.2f}  (max 3)",
                ha='center', transform=ax.transAxes, fontsize=5, color=GREY)
    elif kind == 'match':
        ax.plot([0, 1], [0, 1], color='k', lw=0.5, ls='--', alpha=0.4, zorder=1)
        ax.plot(ow['reward_ratio'], ow['choice_ratio'], color=GREY, lw=1.1, zorder=2)
        ax.plot(ob['reward_ratio'], ob['choice_ratio'], color=BLUE, lw=1.2, zorder=3)
        nature_ax(ax, 'reward ratio', 'choice ratio', nx=3, ny=3)
        ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect('equal')
        ax.set_xticks([0, .5, 1]); ax.set_yticks([0, .5, 1])


In [ ]:
# ---------- column 4: scores across the seven rules (scatter) ----------
# Display order and labels for the scores column (RPE=HR, LPL=HZ).
SCORE_ORDER  = ['HR', 'PC', 'BCM', 'OJA', 'LC', 'COV', 'HZ']
SCORE_LABELS = {'HR': 'RPE', 'PC': 'PC', 'BCM': 'BCM', 'OJA': 'OJA',
                'LC': 'LC', 'COV': 'COV', 'HZ': 'LPL'}

def draw_scores(ax, task, scores, mp=None, mslopes=None):
    order = SCORE_ORDER
    x = np.arange(len(order))
    if task == 'matchingpennies' and mp is not None:
        # two entropies per rule: vs computer (closed) and random reward (open)
        comp = np.array([mp[r]['comp'] for r in order])
        rand = np.array([mp[r]['rand'] for r in order])
        ax.scatter(x, comp, s=22, facecolor='k', edgecolor='k', linewidth=0.5, zorder=3)
        ax.scatter(x, rand, s=22, facecolor='none', edgecolor='k', linewidth=0.7, zorder=3)
        nature_ax(ax, '', '3-bit entropy', nx=len(order), ny=4)
        ax.set_ylim(0, 3); ax.set_yticks([0, 1, 2, 3])
        ax.text(0.0, 1.04, '● vs computer   ○ p(rew)=0.5', transform=ax.transAxes,
                fontsize=4.5, va='bottom', color='k')
    else:
        y = np.array([scores[r] for r in order], float)
        ax.scatter(x, y, s=22, color='k', edgecolor='none', zorder=3)
        ylab = 'max PM index' if task in ('baited', 'unbaited') else 'score'
        nature_ax(ax, '', ylab, nx=len(order), ny=4)
        if task in ('selectivity', 'slowfeature'):
            ax.set_ylim(0, 1)
        elif task in ('baited', 'unbaited'):
            ax.axhline(0, color=GREY, lw=0.4, zorder=1)
            ax.set_ylim(-1, 1); ax.set_yticks([-1, -0.5, 0, 0.5, 1])
        else:
            pad = 0.08 * (np.nanmax(y) - np.nanmin(y) + 1e-9)
            ax.set_ylim(np.nanmin(y) - pad, np.nanmax(y) + pad)
    ax.set_xlim(-0.6, len(order) - 0.4)
    ax.set_xticks(x)
    ax.set_xticklabels([SCORE_LABELS[r] for r in order], fontsize=5.5,
                       fontweight='bold', rotation=45, ha='right')
    ax.tick_params(axis='x', length=0)


In [ ]:
# ---------- config + run all models ----------
SIZE = 'small'
# Each example panel shows the BEST and WORST rule for that task (by the score
# metric). Override 'best_ex'/'worst_ex' to force a specific rule for a row.
TASKS = [
    dict(key='selectivity',     label='Feature\nselectivity',        runner=run_selectivity),
    dict(key='slowfeature',     label='Slow feature\nselectivity',   runner=run_slowfeature),
    dict(key='matchingpennies', label='Costly\nstochasticity',       runner=run_matchingpennies),
    dict(key='baited',          label='Baited bandit\n(prob. match.)',   runner=run_baited),
    dict(key='unbaited',        label='Unbaited bandit\n(prob. match.)', runner=run_unbaited),
]

DATA = {}
for t in TASKS:
    k = t['key']
    scores = rule_scores(k, SIZE)
    br = t.get('best_ex',  best_rule(k, SIZE))
    wr = t.get('worst_ex', worst_rule(k, SIZE))
    if k == 'selectivity':
        # target: LC (the rule BCM-1982 was about); example: PC, a selective seed
        lc_target = run_sel_seeds(best_hparams('selectivity', 'LC', SIZE), n_show=3)
        pc_ex     = run_sel_seeds(best_hparams('selectivity', 'PC', SIZE), n_show=3)
        DATA[k] = dict(scores=scores, best=br, worst=wr,
                       lc_target=lc_target, pc_ex=pc_ex)
        print(f'{k:16s} LC peaks={lc_target["peaks"]} chis={np.round(lc_target["chis"],2)}  '
              f'PC peaks={pc_ex["peaks"]} chis={np.round(pc_ex["chis"],2)}')
    elif k == 'matchingpennies':
        # scores: two entropies per rule (vs computer / random reward); example:
        # best-gap rule, animal-data style.
        mpsc = mp_scores(SIZE); mbr = mp_best_rule(SIZE)
        mp_beh = run_mp_behavior(mp_best_cell(mbr, SIZE))
        DATA[k] = dict(scores=scores, mp_scores=mpsc, best=mbr, mp_beh=mp_beh)
        print(f'{k:16s} best-gap rule={mbr}  gap={mpsc[mbr]["gap"]:.2f} '
              f'(comp={mpsc[mbr]["comp"]:.2f}, rand={mpsc[mbr]["rand"]:.2f})')
    elif k in ('baited', 'unbaited'):
        # scores column = MAX PM index (strongest exploitation regime, read from
        # the stored sweep); examples = high/medium/worst rule by that score,
        # each drawn at its max-index cell (cached curves).
        mm = {r: matching_max_score(k, r, SIZE) for r in RULES}   # scores column
        mc = get_matching(k); rules = mc['rules']; curves = mc['curves']; rr = mc['rr']
        exs = []
        for r in MATCH_RULES:                                     # curves at the shared cell
            cell = matching_shared_cell(r, SIZE)
            idx = cell['bait_idx'] if k == 'baited' else cell['unbait_idx']
            exs.append(dict(rule=r, score=idx, rr=rr, curve=curves[rules.index(r)]))
        DATA[k] = dict(scores=mm, examples=exs)
        print(f'{k:16s} example rules {MATCH_RULES} at shared cells '
              f'(idx {[round(e["score"], 2) for e in exs]})')
    else:
        out_best  = t['runner'](best_hparams(k, br, SIZE))
        out_worst = t['runner'](best_hparams(k, wr, SIZE))
        DATA[k] = dict(scores=scores, best=br, worst=wr,
                       out_best=out_best, out_worst=out_worst)
        print(f'{k:16s} best={br:4s}  worst={wr:4s}')
print('\nall runs complete')


In [ ]:
# ---------- assemble the S1 grid ----------
COLS = ['Task schematic', 'Target behavior', 'Example model behavior', 'Scores across rules']
nR, nC = len(TASKS), len(COLS)

fig, axes = plt.subplots(nR, nC, figsize=(8.4, 1.28 * nR + 0.6),
                         gridspec_kw=dict(wspace=0.42, hspace=0.55))

for ci, ctitle in enumerate(COLS):
    axes[0, ci].set_title(ctitle, fontsize=9, pad=10)

for ri, t in enumerate(TASKS):
    k = t['key']; d = DATA[k]
    draw_schematic(axes[ri, 0], k)
    draw_target(axes[ri, 1], k, d)
    draw_example(axes[ri, 2], k, d)
    draw_scores(axes[ri, 3], k, d['scores'], mp=d.get('mp_scores'), mslopes=d.get('match_slopes'))
    # row label + example-rule tag
    axes[ri, 0].text(-0.42, 0.5, t['label'], transform=axes[ri, 0].transAxes,
                     ha='right', va='center', fontsize=8.5, fontweight='bold', rotation=0)

fig.suptitle('Figure S1  —  Behavior grid: five tasks across seven local rules',
             fontsize=10, y=0.995)
fig.text(0.5, -0.01,
         'Scores column: best performance of each rule (points). Matching pennies shows two entropies: '
         'closed = vs computer opponent, open = random reward p=0.5; baited/unbaited use the max PM index (strongest exploitation / '
         'overmatching regime; +1 = step). Example column: real simulation — feature-selectivity = PC (3 seeds); matching-pennies = '
         'best-gap rule (animal-data style); baited/unbaited = rules COV/BCM/PC/OJA at one shared cell per rule (joint-best over both '
         'bandits, 50k/32 seeds), so a rule differs across the two bandits only by environment; slow-feature = best (blue)/worst (grey). '
         'Feature-selectivity target from LC (BCM-1982 Fig. 4 style).',
         ha='center', fontsize=6, color=GREY)

out_svg = 'S1_behavior_grid.svg'; out_png = 'S1_behavior_grid.png'
fig.savefig(out_svg, bbox_inches='tight')
fig.savefig(out_png, dpi=200, bbox_inches='tight')
print('saved', os.path.abspath(out_svg), '/', out_png)
plt.show()
